# Text-to-Speech with Parler-TTS

Parler-TTS is a controllable text-to-speech model that allows you to generate speech with fine-grained control over voice characteristics using natural language descriptions. Instead of selecting from pre-defined voice IDs, you can describe the desired voice using attributes like gender, speaking style, pace, emotion, and acoustic environment.

This notebook demonstrates how to use Parler-TTS to generate speech with different voice characteristics by providing descriptive prompts alongside your text input.

In [ ]:
# Install required packages
!pip install parler-tts torch soundfile IPython

## Load Model

In [ ]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer
import soundfile as sf
from IPython.display import Audio

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model and tokenizer
model = ParlerTTSForConditionalGeneration.from_pretrained(
    "parler-tts/parler-tts-mini-v1",
    torch_dtype=torch.float16,
).to(device)

tokenizer = AutoTokenizer.from_pretrained("parler-tts/parler-tts-mini-v1")

print(f"Model loaded successfully on device: {device}")

## Generate Speech with Voice Description

In [ ]:
# Define voice description and text to synthesize
voice_description = "A female speaker with a warm, clear voice, speaking at a moderate pace in a quiet room"
text_prompt = "Hello, welcome to Parler-TTS. This is a demonstration of controllable text-to-speech synthesis."

# Tokenize inputs
input_ids = tokenizer(voice_description, return_tensors="pt").input_ids.to(device)
prompt_input_ids = tokenizer(text_prompt, return_tensors="pt").input_ids.to(device)

# Generate audio
generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
audio_arr = generation.cpu().numpy().squeeze()

# Play audio
Audio(audio_arr, rate=model.config.sampling_rate)

## Different Voice Styles

In [ ]:
# Try different voice descriptions
voice_styles = [
    "A male speaker with a deep, authoritative voice, speaking slowly and clearly",
    "A young female speaker with an expressive, energetic voice, speaking quickly with enthusiasm",
    "A calm male speaker with a soothing voice, speaking at a slow pace in a quiet environment",
    "A professional female speaker with a clear, articulate voice, speaking at a fast pace"
]

text = "Artificial intelligence is transforming the way we interact with technology."

for i, voice_desc in enumerate(voice_styles, 1):
    print(f"\nStyle {i}: {voice_desc}")
    
    # Tokenize inputs
    input_ids = tokenizer(voice_desc, return_tensors="pt").input_ids.to(device)
    prompt_input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)
    
    # Generate audio
    generation = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)
    audio_arr = generation.cpu().numpy().squeeze()
    
    # Display audio player
    display(Audio(audio_arr, rate=model.config.sampling_rate))

## Batch Generation

In [ ]:
# Generate multiple utterances in a batch
voice_descriptions = [
    "A female speaker with a warm voice",
    "A male speaker with a deep voice"
]

text_prompts = [
    "This is the first sentence.",
    "This is the second sentence."
]

# Tokenize batch inputs
input_ids = tokenizer(voice_descriptions, return_tensors="pt", padding=True).input_ids.to(device)
prompt_input_ids = tokenizer(text_prompts, return_tensors="pt", padding=True).input_ids.to(device)

# Generate audio for batch
generations = model.generate(input_ids=input_ids, prompt_input_ids=prompt_input_ids)

# Play each generated audio
for i, generation in enumerate(generations):
    print(f"\nAudio {i+1}: {text_prompts[i]}")
    audio_arr = generation.cpu().numpy()
    display(Audio(audio_arr, rate=model.config.sampling_rate))

## Tips

### Voice Description Guidelines
When crafting voice descriptions, you can control multiple aspects:

- **Gender**: male, female
- **Tone**: warm, cold, friendly, professional, authoritative, soothing
- **Pace**: slow, moderate, fast, speaking slowly/quickly
- **Clarity**: clear, muffled, articulate
- **Emotion**: expressive, monotone, enthusiastic, calm, excited
- **Environment**: quiet room, noisy background, echoey space
- **Age**: young, mature, elderly (implicit)

### Generation Parameters
You can also control generation with additional parameters:

```python
generation = model.generate(
    input_ids=input_ids,
    prompt_input_ids=prompt_input_ids,
    temperature=1.0,           # Controls randomness (0.0-2.0)
    max_length=None,           # Maximum length of generated audio
    do_sample=True,            # Whether to use sampling
)
```

### Best Practices
- Be specific but concise in voice descriptions
- Combine multiple attributes for better control
- Experiment with different descriptions to find the best match
- The model works best with natural, conversational descriptions
- For consistent results, use similar description patterns